In [1]:
!pip install ultralytics
!pip install tqdm
!pip install torchmetrics
!pip install pycocotools
!pip install faster-coco-eval

In [2]:
import os
import time
from pathlib import Path
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from torchvision.models.detection import ssd300_vgg16
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Google Colab SSD Training


In [ ]:
# Google Colab - 01
from pathlib import Path
from google.colab import drive
import zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/Colab Notebooks/final_unified_dataset.zip"

extract_root = Path("/content")
dataset_path = extract_root / "final_unified_dataset"
if not dataset_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print(f"Extracted dataset to {extract_root}")
else:
    print(f"Dataset already extracted at {extract_root}")

root = extract_root

print("\nDataset summary:")
print("Train images folder exists:", (root / "final_unified_dataset/images/train").exists())
print("Val images folder exists:", (root / "final_unified_dataset/images/val").exists())
print("Test images folder exists:", (root / "final_unified_dataset/images/test").exists())
print("Train labels folder exists:", (root / "final_unified_dataset/labels/train").exists())
print("Val labels folder exists:", (root / "final_unified_dataset/labels/val").exists())
print("Test labels folder exists:", (root / "final_unified_dataset/labels/test").exists())
print("YAML exists:", (root / "final_unified_dataset/dataset.yaml").exists())

for split in ["train", "val", "test"]:
    img_dir = root / "final_unified_dataset/images" / split
    lbl_dir = root / "final_unified_dataset/labels" / split

    num_images = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    num_labels = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    print(f"\n{split.capitalize()} split:")
    print(f"  Images: {num_images}")
    print(f"  Labels: {num_labels}")

In [ ]:
# Google Colab - 02 - SSD model setup
class SSDDataset(Dataset):
    def __init__(self, root, split="train"):
        self.root = Path(root)
        self.img_dir = self.root / "images" / split
        self.label_dir = self.root / "labels" / split

        if not self.img_dir.exists():
            raise FileNotFoundError(f"Image directory {self.img_dir} not found")
        if not self.label_dir.exists():
            raise FileNotFoundError(f"Label directory {self.label_dir} not found")

        self.images = sorted(list(self.img_dir.glob("*.jpg")) + list(self.img_dir.glob("*.png")))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / f"{img_path.stem}.txt"
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Could not read image {img_path}")
        h, w = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        boxes = []
        labels = []
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.strip().split())
                    # Convert YOLO normalized format to pixel coordinates
                    x_min = (x - bw / 2) * w
                    y_min = (y - bh / 2) * h
                    x_max = (x + bw / 2) * w
                    y_max = (y + bh / 2) * h
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)  # +1 for SSD background class

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        # Convert image to tensor [C,H,W] and normalize
        image = torch.tensor(image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return image, target

In [26]:
# Google Colab - 03 - SSD Training (Colab) with progress bar + precision/recall
import time
import torch
from torchvision.models.detection import ssd300_vgg16
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Colab dataset path
root = Path("/content/final_unified_dataset")

train_dataset = SSDDataset(root, "train")
val_dataset = SSDDataset(root, "val")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                        collate_fn=lambda x: tuple(zip(*x)))

# GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = ssd300_vgg16(weights="DEFAULT")
model.head.classification_head.num_classes = 3  # 2 classes + background
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2]-box1[0]) * (box1[3]-box1[1])
    box2_area = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def batch_precision_recall(outputs, targets, iou_threshold=0.5):
    correct = 0
    total_pred = 0
    total_gt = 0

    for out, tgt in zip(outputs, targets):
        pred_boxes = out['boxes'].detach().cpu()
        gt_boxes = tgt['boxes'].detach().cpu()
        total_pred += len(pred_boxes)
        total_gt += len(gt_boxes)

        for p in pred_boxes:
            if any(compute_iou(p, g) > iou_threshold for g in gt_boxes):
                correct += 1

    precision = correct / total_pred if total_pred > 0 else 0
    recall = correct / total_gt if total_gt > 0 else 0
    return precision, recall

# Training loop 
start_time_total = time.time()
epochs = 50

for epoch in range(epochs):
    epoch_start = time.time()
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.3f}"})

    model.eval()
    precisions = []
    recalls = []

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            p, r = batch_precision_recall(outputs, targets)
            precisions.append(p)
            recalls.append(r)

    epoch_precision = sum(precisions) / len(precisions)
    epoch_recall = sum(recalls) / len(recalls)
    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss:.4f} | "
          f"Precision: {epoch_precision:.3f} | Recall: {epoch_recall:.3f} | "
          f"Epoch Time: {epoch_time:.2f}s")

total_time = time.time() - start_time_total
print(f"\nTotal training time: {total_time/60:.2f} minutes")

ValueError: num_samples should be a positive integer value, but got num_samples=0